In [16]:
"""
Generate paired sets of simulated echelle CCD exposures.
The script creates 15 short-exposure and 15 long-exposure FITS files.
"""

import os
from datetime import datetime, timedelta, timezone

import numpy as np
from astropy.io import fits
from scipy.interpolate import CubicSpline
from scipy.ndimage import convolve


# ============================================================================
# 1. OUTPUT SETTINGS
# ============================================================================

DIR_CCD1 = "/Volumes/SCRATCH4TB/111111/0_1"
DIR_CCD2 = "/Volumes/SCRATCH4TB/111111/1_0"

# True  -> Data with charge sharing
# False -> Data without charge sharing
USE_CHARGE_SHARING = True

# True  -> Non-linear
# False -> Linear
USE_NONLINEARITY = True

MASK_FILE = ("/Users/z5517274/Priyash/Non-Linearity Correction/Masks/region_mask_mask.fits")

N_EXP = 15
OVERWRITE_EXISTING = True


# ============================================================================
# 2. EXPOSURE AND SIGNAL-SCALING SETTINGS
# ============================================================================

EXP_TIME_CCD1 = 0.1  # seconds; value written to the FITS header
EXP_TIME_CCD2 = 1.0  # seconds; value written to the FITS header

# These factors are intentionally retained from the original simulation.
# They imply a signal ratio of 10 / 0.35 = 28.57, although the exposure-time
# ratio in the FITS headers is 1.0 / 0.1 = 10.
#
# With MAX_SIGNAL = 65535 ADU, the nominal CCD1 peak before noise is
# 0.35 * 65535 = 22937 ADU. Increase CCD1_SIGNAL_FACTOR only if a nominal
# simulated peak closer to 35000 ADU is required.
CCD1_SIGNAL_FACTOR = 0.35
CCD2_SIGNAL_FACTOR = 10.0


# ============================================================================
# 3. CHARGE-SHARING SETTINGS
# ============================================================================

F_MIN = 0.001
F_MAX = 0.10 #0.12
SIGMOID_CENTER = 40000 #35000.0  # ADU
SIGMOID_WIDTH = 10000 #12000.0   # ADU
SIGMOID_POWER = 0.6 #0.80

# True redistributes charge to all eight surrounding pixels.
# False redistributes it only to the four edge-sharing pixels.
USE_8_NEIGHBOURS = True

# Fixed pixel-column variation in the sharing fraction.
COLUMN_VARIATION_SIGMA = 0.005
COLUMN_VARIATION_SEED = 123

# The convolution uses zero outside the active array. Therefore, charge that
# diffuses beyond a physical CCD boundary is lost, as can occur at the edge of
# a real detector. Interior pixels conserve redistributed charge.
SHARING_BOUNDARY_MODE = "constant"
SHARING_BOUNDARY_VALUE = 0.0


# ============================================================================
# 4. DETECTOR NON-LINEARITY SETTINGS
# ============================================================================

# Detector response:
# measured_ADU = NONLINEAR_GAIN * (
#     NONLINEAR_A1 * signal
#     + NONLINEAR_A2 * signal**2
#     + NONLINEAR_A3 * signal**3
# )
#
# These coefficients are retained exactly from the supplied non-linear
# simulation. The response is applied after charge sharing (when enabled) and
# before read noise and final clipping.
NONLINEAR_GAIN = 1.0
NONLINEAR_A1 = 1.0
NONLINEAR_A2 = -5.0e-7
NONLINEAR_A3 = 1.0e-12


# ============================================================================
# 5. DETECTOR AND NOISE SETTINGS
# ============================================================================

NY = 4112
NX = 4202

# Gain is assumed to be 1 electron/ADU, so Poisson counts, read noise and the
# saved image values use the same numerical units.
GAIN_E_PER_ADU = 1.0
READOUT_NOISE_ADU = 4.0
MAX_COUNTS_ADU = 65535.0
MAX_SIGNAL_ADU = 65535.0

# A fixed expected background is added to every pixel before Poisson sampling.
# It is deliberately per exposure rather than per second, consistent with the
# original CONFIG value. Change this if a time-scaling background is required.
BACKGROUND_LEVEL_ADU = 0.0 #4.0

# Negative values after read-noise addition are clipped to zero. With a 4 ADU
# background this bias is much smaller than it would be for a zero background.
CLIP_MIN_ADU = 0.0


# ============================================================================
# 6. ORDER-PATTERN SETTINGS
# ============================================================================

RIPPLE_AMPLITUDE = 0.45
N_RIPPLE_PEAKS = 18
EDGE_SOFTNESS_PIXELS = 0.60

# Two-dimensional illumination envelope.
X_PEAK_FRACTION = 0.30
Y_PEAK_FRACTION = 0.45
X_SCALE_FRACTION = 0.18
Y_SCALE_FRACTION = 0.15


# ============================================================================
# 7. RANDOM SEEDS AND TIMING SETTINGS
# ============================================================================

PATTERN_SEED = 0
CCD1_SEED = 0
CCD2_SEED = 1

# Time between the end of one exposure and the start of the next.
READOUT_GAP_SECONDS = 0.0


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def load_order_mask(mask_file, expected_shape):
    """Load a binary order mask and verify that it matches the CCD shape."""

    with fits.open(mask_file, memmap=True) as hdul:
        mask = np.asarray(hdul[0].data)

    if mask.shape != expected_shape:
        raise ValueError(
            f"Mask shape {mask.shape} does not match expected CCD shape "
            f"{expected_shape}."
        )

    return mask == 1


def sigmoid_sharing_fraction(
    signal,
    f_min,
    f_max,
    centre,
    width,
    power,
):
    """Return the signal-dependent fraction redistributed to neighbours."""

    if width <= 0:
        raise ValueError("SIGMOID_WIDTH must be greater than zero.")
    if power <= 0:
        raise ValueError("SIGMOID_POWER must be greater than zero.")
    if not 0.0 <= f_min <= f_max <= 1.0:
        raise ValueError("Require 0 <= F_MIN <= F_MAX <= 1.")

    scaled_signal = np.clip((signal - centre) / width, -50.0, 50.0)
    logistic = 1.0 / (1.0 + np.exp(-scaled_signal))
    return f_min + (f_max - f_min) * logistic**power


def make_sharing_kernel(use_8_neighbours):
    """Construct a normalised charge-redistribution kernel."""

    if use_8_neighbours:
        kernel = np.array(
            [
                [0.02, 0.03, 0.02],
                [0.03, 0.00, 0.03],
                [0.02, 0.03, 0.02],
            ],
            dtype=float,
        )
    else:
        kernel = np.array(
            [
                [0.00, 0.25, 0.00],
                [0.25, 0.00, 0.25],
                [0.00, 0.25, 0.00],
            ],
            dtype=float,
        )

    return kernel / kernel.sum()


def random_peaked_fluctuation(x, n_peaks, amplitude, rng):
    """Generate a smooth, non-periodic one-dimensional fluctuation."""

    x_peaks = np.linspace(x.min(), x.max(), n_peaks)
    spacing = x_peaks[1] - x_peaks[0]
    x_peaks += rng.uniform(-0.35, 0.35, n_peaks) * spacing
    y_peaks = 1.0 + amplitude * rng.uniform(-1.0, 1.0, n_peaks)

    spline = CubicSpline(x_peaks, y_peaks, bc_type="natural")
    return spline(x)


def flat_top_fluctuated_profile(
    x,
    centre,
    core_width,
    ripple_amplitude,
    n_peaks,
    edge_softness,
    rng,
):
    """Construct one sharp-edged, fluctuated order cross-section."""

    offset = x - centre

    stripe = 0.5 * (
        np.tanh((offset + core_width / 2.0) / edge_softness)
        - np.tanh((offset - core_width / 2.0) / edge_softness)
    )

    core_mask = np.abs(offset) <= core_width / 2.0

    if np.any(core_mask):
        core_x = offset[core_mask]
        fluctuation = np.zeros_like(core_x, dtype=float)
        segment_width = core_width / n_peaks

        for peak_index in range(n_peaks):
            amplitude = rng.uniform(-ripple_amplitude, ripple_amplitude)
            amplitude = abs(amplitude) if peak_index % 2 == 0 else -abs(amplitude)

            peak_centre = (
                -core_width / 2.0
                + (peak_index + 0.5) * segment_width
            )

            contribution = np.clip(
                1.0
                - np.abs(core_x - peak_centre) / (segment_width / 2.0),
                0.0,
                1.0,
            )

            fluctuation += amplitude * contribution

        stripe[core_mask] *= np.clip(1.0 + fluctuation, 0.7, 1.3)

    return np.clip(stripe, 0.0, None)


def build_order_pattern(order_mask):
    """Build the fixed, noise-free echelle illumination pattern."""

    y, x = np.indices((NY, NX))

    x_centre = NX * X_PEAK_FRACTION
    y_centre = NY * Y_PEAK_FRACTION
    x_scale = NX * X_SCALE_FRACTION
    y_scale = NY * Y_SCALE_FRACTION

    envelope = np.exp(-((x - x_centre) ** 2) / (2.0 * x_scale**2))
    envelope *= np.exp(-((y - y_centre) ** 2) / (2.0 * y_scale**2))
    envelope = np.clip(envelope, 0.0, 1.0)

    pattern = np.zeros((NY, NX), dtype=float)
    rng = np.random.default_rng(PATTERN_SEED)

    for row in range(NY):
        illuminated_columns = np.flatnonzero(order_mask[row])

        if illuminated_columns.size == 0:
            continue

        split_locations = np.flatnonzero(np.diff(illuminated_columns) != 1) + 1
        segments = np.split(illuminated_columns, split_locations)

        for segment in segments:
            start = int(segment[0])
            end = int(segment[-1])
            width = end - start + 1
            local_x = np.arange(width, dtype=float)

            stripe = flat_top_fluctuated_profile(
                x=local_x,
                centre=width / 2.0,
                core_width=float(width),
                ripple_amplitude=RIPPLE_AMPLITUDE,
                n_peaks=N_RIPPLE_PEAKS,
                edge_softness=EDGE_SOFTNESS_PIXELS,
                rng=rng,
            )

            pattern[row, start : end + 1] = stripe

    pattern *= envelope
    pattern *= MAX_SIGNAL_ADU

    return pattern


def draw_poisson(expected_adu, rng):
    """Draw photon counts assuming a gain of 1 electron/ADU."""

    expected_electrons = np.clip(
        expected_adu * GAIN_E_PER_ADU,
        0.0,
        None,
    )
    electrons = rng.poisson(expected_electrons)
    return electrons.astype(float) / GAIN_E_PER_ADU


def apply_charge_sharing(
    realised_adu,
    reference_signal_adu,
    sharing_kernel,
    column_variation,
):
    """Redistribute a signal-dependent fraction of the realised charge."""

    sharing_fraction = sigmoid_sharing_fraction(
        signal=reference_signal_adu,
        f_min=F_MIN,
        f_max=F_MAX,
        centre=SIGMOID_CENTER,
        width=SIGMOID_WIDTH,
        power=SIGMOID_POWER,
    )

    sharing_fraction *= column_variation[np.newaxis, :]
    sharing_fraction = np.clip(sharing_fraction, 0.0, 1.0)

    lost_charge = realised_adu * sharing_fraction
    redistributed_charge = convolve(
        lost_charge,
        sharing_kernel,
        mode=SHARING_BOUNDARY_MODE,
        cval=SHARING_BOUNDARY_VALUE,
    )

    return realised_adu - lost_charge + redistributed_charge


def add_read_noise(frame_adu, rng):
    """Add one independent Gaussian read-noise realisation."""

    return frame_adu + rng.normal(
        loc=0.0,
        scale=READOUT_NOISE_ADU,
        size=frame_adu.shape,
    )


def apply_detector_nonlinearity(signal_adu):
    """Apply the polynomial detector response to the signal."""

    return NONLINEAR_GAIN * (
        NONLINEAR_A1 * signal_adu
        + NONLINEAR_A2 * signal_adu**2
        + NONLINEAR_A3 * signal_adu**3
    )


def simulate_exposure(
    pattern,
    signal_factor,
    rng,
    sharing_kernel,
    column_variation,
):
    """Generate one simulated CCD exposure."""

    expected_source_adu = np.clip(pattern * signal_factor, 0.0, None)
    expected_total_adu = expected_source_adu + BACKGROUND_LEVEL_ADU

    frame_adu = draw_poisson(expected_total_adu, rng)

    if USE_CHARGE_SHARING:
        frame_adu = apply_charge_sharing(
            realised_adu=frame_adu,
            reference_signal_adu=expected_total_adu,
            sharing_kernel=sharing_kernel,
            column_variation=column_variation,
        )

    if USE_NONLINEARITY:
        frame_adu = apply_detector_nonlinearity(frame_adu)

    frame_adu = add_read_noise(frame_adu, rng)

    return np.clip(frame_adu, CLIP_MIN_ADU, MAX_COUNTS_ADU)


def save_fits(data, path, exposure_time, signal_factor, index, obs_time):
    """Save one simulated exposure and its key settings."""

    hdu = fits.PrimaryHDU(data.astype(np.float32))
    header = hdu.header

    header["EXPTIME"] = (exposure_time, "Exposure time [s]")
    header["SIGFACT"] = (signal_factor, "Applied source-signal factor")
    header["FRAMEID"] = (index, "Frame number within exposure set")
    header["DATE-OBS"] = (obs_time.isoformat(), "UTC exposure start")
    header["CCDNAME"] = "SIMULATED"
    header["BUNIT"] = "ADU"
    header["CHSHARE"] = (USE_CHARGE_SHARING, "Charge sharing enabled")
    header["FMIN"] = F_MIN
    header["FMAX"] = F_MAX
    header["FCENTRE"] = SIGMOID_CENTER
    header["FWIDTH"] = SIGMOID_WIDTH
    header["FPOWER"] = SIGMOID_POWER
    header["NONLIN"] = (USE_NONLINEARITY, "Detector non-linearity enabled")
    header["NLGAIN"] = NONLINEAR_GAIN
    header["NLA1"] = NONLINEAR_A1
    header["NLA2"] = NONLINEAR_A2
    header["NLA3"] = NONLINEAR_A3
    header["RNOISE"] = (READOUT_NOISE_ADU, "Read noise [ADU]")
    header["BACKGRND"] = (BACKGROUND_LEVEL_ADU, "Expected background [ADU]")
    header["GAIN"] = (GAIN_E_PER_ADU, "Detector gain [electron/ADU]")

    hdu.writeto(path, overwrite=OVERWRITE_EXISTING)


def generate_exposure_set(
    output_directory,
    filename_prefix,
    exposure_time,
    signal_factor,
    rng,
    pattern,
    sharing_kernel,
    column_variation,
    start_time,
):
    """Generate and save all exposures in one set."""

    observation_time = start_time

    for index in range(1, N_EXP + 1):
        final_frame = simulate_exposure(
            pattern=pattern,
            signal_factor=signal_factor,
            rng=rng,
            sharing_kernel=sharing_kernel,
            column_variation=column_variation,
        )

        filename = os.path.join(
            output_directory,
            f"{filename_prefix}_exp_{index:03d}.fits",
        )

        save_fits(
            data=final_frame,
            path=filename,
            exposure_time=exposure_time,
            signal_factor=signal_factor,
            index=index,
            obs_time=observation_time,
        )

        print(f"Saved {filename}")

        observation_time += timedelta(
            seconds=exposure_time + READOUT_GAP_SECONDS
        )

    return observation_time


# ============================================================================
# MAIN
# ============================================================================

def main():
    os.makedirs(DIR_CCD1, exist_ok=True)
    os.makedirs(DIR_CCD2, exist_ok=True)

    order_mask = load_order_mask(MASK_FILE, expected_shape=(NY, NX))
    pattern = build_order_pattern(order_mask)

    sharing_kernel = make_sharing_kernel(USE_8_NEIGHBOURS)

    column_rng = np.random.default_rng(COLUMN_VARIATION_SEED)
    column_variation = 1.0 + COLUMN_VARIATION_SIGMA * column_rng.normal(
        size=NX
    )

    rng_ccd1 = np.random.default_rng(CCD1_SEED)
    rng_ccd2 = np.random.default_rng(CCD2_SEED)

    current_time = datetime.now(timezone.utc)

    print(f"Charge sharing enabled: {USE_CHARGE_SHARING}")
    print(f"Detector non-linearity enabled: {USE_NONLINEARITY}")
    print(
        "Header exposure-time ratio "
        f"(CCD2/CCD1): {EXP_TIME_CCD2 / EXP_TIME_CCD1:.2f}"
    )
    print(
        "Applied signal-factor ratio "
        f"(CCD2/CCD1): {CCD2_SIGNAL_FACTOR / CCD1_SIGNAL_FACTOR:.2f}"
    )
    print()

    current_time = generate_exposure_set(
        output_directory=DIR_CCD1,
        filename_prefix="ccd1",
        exposure_time=EXP_TIME_CCD1,
        signal_factor=CCD1_SIGNAL_FACTOR,
        rng=rng_ccd1,
        pattern=pattern,
        sharing_kernel=sharing_kernel,
        column_variation=column_variation,
        start_time=current_time,
    )

    generate_exposure_set(
        output_directory=DIR_CCD2,
        filename_prefix="ccd2",
        exposure_time=EXP_TIME_CCD2,
        signal_factor=CCD2_SIGNAL_FACTOR,
        rng=rng_ccd2,
        pattern=pattern,
        sharing_kernel=sharing_kernel,
        column_variation=column_variation,
        start_time=current_time,
    )

    print()
    print(f"All {2 * N_EXP} exposures saved successfully.")


if __name__ == "__main__":
    main()

Charge sharing enabled: True
Detector non-linearity enabled: True
Header exposure-time ratio (CCD2/CCD1): 10.00
Applied signal-factor ratio (CCD2/CCD1): 28.57

Saved /Volumes/SCRATCH4TB/311114/0_1/ccd1_exp_001.fits
Saved /Volumes/SCRATCH4TB/311114/0_1/ccd1_exp_002.fits
Saved /Volumes/SCRATCH4TB/311114/0_1/ccd1_exp_003.fits
Saved /Volumes/SCRATCH4TB/311114/0_1/ccd1_exp_004.fits
Saved /Volumes/SCRATCH4TB/311114/0_1/ccd1_exp_005.fits
Saved /Volumes/SCRATCH4TB/311114/0_1/ccd1_exp_006.fits
Saved /Volumes/SCRATCH4TB/311114/0_1/ccd1_exp_007.fits
Saved /Volumes/SCRATCH4TB/311114/0_1/ccd1_exp_008.fits
Saved /Volumes/SCRATCH4TB/311114/0_1/ccd1_exp_009.fits
Saved /Volumes/SCRATCH4TB/311114/0_1/ccd1_exp_010.fits
Saved /Volumes/SCRATCH4TB/311114/0_1/ccd1_exp_011.fits
Saved /Volumes/SCRATCH4TB/311114/0_1/ccd1_exp_012.fits
Saved /Volumes/SCRATCH4TB/311114/0_1/ccd1_exp_013.fits
Saved /Volumes/SCRATCH4TB/311114/0_1/ccd1_exp_014.fits
Saved /Volumes/SCRATCH4TB/311114/0_1/ccd1_exp_015.fits
Saved /Volumes/